# Chilli Plant Disease Detection — Training Notebook
Versions pinned to match local backend: **TensorFlow 2.16.1 · NumPy 1.26.4 · Pillow 10.3.0**

### Run order
1. Run **Cell 1** — it installs packages then **restarts the runtime automatically**
2. After restart, run **Cells 2 → 10** in order (Cell 1 does not need to run again)

In [ ]:
# ── 1. Install pinned dependencies + kaggle ──────────────────────────────────
# Conflict warnings from other Colab packages are safe to ignore.
# Runtime will restart automatically after install — that is expected.
!pip install -q tensorflow==2.16.1 numpy==1.26.4 pillow==10.3.0 kaggle==1.6.17

import os
os.kill(os.getpid(), 9)  # force restart so pinned versions are active

In [ ]:
# ── 2. Configure Kaggle API credentials ──────────────────────────────────────
# Get your key: https://www.kaggle.com/settings → API → Create New Token
import os, json

KAGGLE_USERNAME = 'your_kaggle_username'  # ← replace
KAGGLE_KEY      = 'your_kaggle_api_key'   # ← replace

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
# Also set env vars so subprocess inherits them (prevents 403)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY
print('Kaggle credentials configured.')

In [ ]:
# ── 3. Download & extract dataset ────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/zaidkhan0997/chilli-plant-disease-detection-dataset
import pathlib, subprocess

DATASET_SLUG = 'zaidkhan0997/chilli-plant-disease-detection-dataset'

result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG,
     '-p', '/content/chilli_data', '--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Find the deepest directory whose immediate children are all class folders
root = pathlib.Path('/content/chilli_data')
def has_only_class_dirs(p):
    children = list(p.iterdir())
    return len(children) > 1 and all(c.is_dir() for c in children)

candidates = sorted([p for p in root.rglob('*') if p.is_dir() and has_only_class_dirs(p)])
DATASET_DIR = str(candidates[0]) if candidates else str(root)

print('Dataset dir :', DATASET_DIR)
print('Classes found:', sorted([p.name for p in pathlib.Path(DATASET_DIR).iterdir() if p.is_dir()]))

In [ ]:
# ── 4. Mount Google Drive (for saving model output) ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 5. Load & split dataset ──────────────────────────────────────────────────
import tensorflow as tf
import numpy as np

IMG_SIZE   = (64, 64)  # must match model.py input shape
BATCH_SIZE = 32
EPOCHS     = 20
SEED       = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2, subset='training',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2, subset='validation',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print('Classes    :', CLASS_NAMES)
print('Num classes:', NUM_CLASSES)

In [ ]:
# ── 6. Normalise + prefetch ──────────────────────────────────────────────────
normalise = tf.keras.layers.Rescaling(1./255)
train_ds  = train_ds.map(lambda x, y: (normalise(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds    = val_ds.map(lambda x, y:   (normalise(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# ── 7. Build model — identical architecture to backend/model.py ──────────────
model = tf.keras.Sequential([
    tf.keras.Input(shape=(64, 64, 3)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ── 8. Train ─────────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

In [ ]:
# ── 9. Plot training curves ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── 10. Save model.h5 to Drive ───────────────────────────────────────────────
SAVE_PATH = '/content/drive/MyDrive/chilli_model.h5'
model.save(SAVE_PATH)
print('Model saved to', SAVE_PATH)

In [ ]:
# ── 11. Print CLASS_LABELS — copy & paste into backend/model.py ──────────────
print('CLASS_LABELS = [')
for name in CLASS_NAMES:
    print(f'    "{name}",')
print(']')